In [0]:
%sql
create schema if not exists sayli_adminenqurious_onmicrosoft_com.gcp_bronze;

In [0]:
mount_name = "gbmartmsklsayli"

In [0]:
catalog_name = "sayli_adminenqurious_onmicrosoft_com"
schema_name = "gcp_bronze"

In [0]:
# creating a list of all the tables (csv files) in the mounted bucket
csv_files = dbutils.fs.ls(f"dbfs:/mnt/{mount_name}")
Tables = [file.name[:-4] for file in csv_files if file.name.endswith(".csv")]
print(Tables)

In [0]:
# Step 1: Create Delta Tables with Column Mapping Mode
for table in Tables:
    create_table_query = f"""
    CREATE TABLE IF NOT EXISTS {catalog_name}.{schema_name}.{table}
    TBLPROPERTIES ('delta.columnMapping.mode' = 'name')  -- Enable column mapping for schema evolution
    """
    spark.sql(create_table_query)

In [0]:
%sql
show tables in sayli_adminenqurious_onmicrosoft_com.gcp_bronze;

In [0]:
%sql
select * from sayli_adminenqurious_onmicrosoft_com.gcp_bronze.customers;

In [0]:
addresses_df = spark.read.csv(f"dbfs:/mnt/{mount_name}/addresses.csv", header=True, inferSchema=True)
addresses_df.display()

In [0]:
for table in Tables:
    raw_data_path = f"dbfs:/mnt/{mount_name}/{table}.csv"

    copy_into_query = f"""
    COPY INTO {catalog_name}.{schema_name}.{table}
    FROM '{raw_data_path}'
    FILEFORMAT = CSV
    FORMAT_OPTIONS(
        "header" = "true",
        "inferSchema" = "true",
        "mergeSchema" = "true",
        "timestampFormat" = "dd-MM-yyyy HH.mm",
        "multiLine" = "true",  -- Fix multiline data issues
        "quote" = '"',  -- Properly handle quoted values
        "escape" = '"'  -- Escape any embedded quotes correctly
    )
    COPY_OPTIONS("mergeSchema" = "true")
    """
    spark.sql(copy_into_query)

print("Data successfully loaded into Delta Bronze tables with fixed parsing.")

In [0]:
%sql
select * from sayli_adminenqurious_onmicrosoft_com.gcp_bronze.addresses;
+

In [0]:
%sql
describe history sayli_adminenqurious_onmicrosoft_com.gcp_bronze.addresses;

In [0]:
%sql
ALTER TABLE amazon.bronze_ecom.orders SET TBLPROPERTIES(delta.enableChangeDataFeed = true)